# NB 57 — Subspace steering: pooled vs hierarchical (multilingual)

The multilingual twin of NB 56 (English-only): identical format, but the concept
subspaces are built from lemmas in the **full FRH language set** used for Llama
(`SupportedLanguages.Llama`, 8 languages) at the original `MIN_LEMMAS = 11` — the
convention the validated pool (NB 32) was actually built under.

Builds the validated-pool concepts (NB 32) with **both** `subspace/` concept
constructions and compares them under two regimes:

**A. Singles** — single-concept guided generation, one concept at a time, swept
over the fluency dial `λ ∈ {0, 0.4}` (as in NB 44).
- **pooled** (`method="pooled"`) — one vote per token.
- **hierarchical** (`method="hierarchical"`) — one vote per lemma.

**B. Blend pairs (multi-concept composition)** — mirrors the FRH multi-concept
steering experiments (NB 40/43): concept *pairs* are blended into a single guide
via `ConceptSubspace.blend([cA, cB])` and steered at the fluency-dial operating
points `λ ∈ {0, 0.4}`. Success is the **joint** metric — both members must
improve (Δ>0), joint Δ = `min` over members — exactly as in the frame pipeline.
The blend arm is run under *both* constructions, so the pooled-vs-hierarchical
comparison carries through to composition.

Both regimes share the recipe: `k=4, steps=24, max_tokens=3`,
Llama-3.1-8B-Instruct bf16, pool `resources/32_concept_pool_bf16.json`, the
canonical FRH pairs `resources/33_scaling_groups_bf16.json`, the 10 canonical
prompts (NB 44's set), fixed subspace knobs `RANK=8`, `R_LEMMA=1`. Metrics come
from the shared `EvaluationHarness` (expression Δ, presence success, ppl ratio).

**Lemma language: 8-language FRH set** (`SupportedLanguages.Llama`) at
`MIN_LEMMAS = 11` — the same language coverage and threshold the FRH multi-concept
experiments and the validated pool (NB 32) used. Any pool / pair member below the
threshold is skipped (rare here, since the pool was selected under this setting).

Set `SMOKE = False` to run the full pool + all pairs; `SMOKE = True` (default)
runs a small subset for a quick end-to-end check.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.nlp import SupportedLanguages
from subspace import ConceptSubspace, SubspaceUnembeddingRepresentation

In [ ]:
# 8-language FRH set (SupportedLanguages.Llama) at the pool's original threshold.
MIN_LEMMAS = 11
MAX_TOKENS, K, STEPS = 3, 4, 24
RANK, R_LEMMA = 8, 1
LAMBDAS = [0.0, 0.4]  # fluency dial operating points (blend arm), as in NB 40/44
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"
ARMS = ["pooled", "hierarchical"]

SMOKE = True  # subset + `_smoke` outputs; flip to False for the full run

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
CONCEPTS = [r["synset"] for r in POOL["pool"]]  # singles (40 concepts)
GROUPS = json.loads(Path("resources/33_scaling_groups_bf16.json").read_text())["pairs"]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

if SMOKE:
    CONCEPTS, QUESTIONS = CONCEPTS[:2], QUESTIONS[:2]
    # Pair the two available smoke concepts (keeps the blend section non-empty
    # even if a canonical FRH pair member happens to fall below the threshold).
    GROUPS = [{"a": CONCEPTS[0], "b": CONCEPTS[1], "pair_rho": None}]

SUFFIX = "_smoke" if SMOKE else ""
LOG_PATH = Path(f"resources/57_subspace_ml{SUFFIX}.jsonl")
LOG_PATH.unlink(missing_ok=True)  # fresh run; harness appends
print(
    f"singles: {len(CONCEPTS)} concepts | "
    f"blend: {len(GROUPS)} pairs x {len(LAMBDAS)} lambda | "
    f"{len(ARMS)} arms x {len(QUESTIONS)} prompts"
)

In [ ]:
sur = SubspaceUnembeddingRepresentation.from_model_id(
    MODEL_ID,
    rank=RANK,
    r_lemma=R_LEMMA,
    language_codes=SupportedLanguages.Llama,  # full 8-language FRH set
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
prompts = [chat(u, a) for u, a in QUESTIONS]

# Baseline is arm-independent (model only) — generate once, reuse everywhere.
baseline_texts = sur.generate(prompts, max_new_tokens=STEPS + 2, do_sample=False)

for arm in ARMS:
    # A fresh harness per arm: its expression cache is keyed by (text, synset)
    # only, and both arms share synset names with different bases — a shared
    # harness would return one arm's cached baseline expression for the other.
    harness = EvaluationHarness(sur, LOG_PATH, MIN_LEMMAS, MAX_TOKENS)
    harness.member_lemmas(CONCEPTS[0])  # force the WordNet parquet load out of loops

    built = {}

    def concept(name):
        """Build (once per arm) the subspace for a synset, or None if unavailable."""
        if name not in built:
            try:
                built[name] = sur.get_concept_cached(
                    name, MIN_LEMMAS, MAX_TOKENS, method=arm
                )
            except ValueError:
                print(f"  skip {name}: below min_lemmas={MIN_LEMMAS} in this language set")
                built[name] = None
        return built[name]

    # -- A. singles (fluency dial λ ∈ LAMBDAS) -----------------------------
    for name in CONCEPTS:
        c = concept(name)
        if c is None:
            continue
        for lam in LAMBDAS:
            torch.cuda.empty_cache()
            texts, probe = sur.generate_with_topk_guide(
                prompts, guide=c, k=K, steps=STEPS, fluency_weight=lam
            )
            harness.evaluate(
                prompts,
                texts,
                [name],
                config={
                    "mode": "single",
                    "model": MODEL_KEY,
                    "arm": arm,
                    "concept": name,
                    "lam": lam,
                    "rank": RANK,
                    "r_lemma": R_LEMMA,
                    "k": K,
                    "steps": STEPS,
                },
                probe=probe,
                baseline_texts=baseline_texts,
                concepts={name: c},
                angles=False,  # angle metrics read concept.tensor (frame-only)
            )

    # -- B. blend pairs (multi-concept composition) ------------------------
    for grp in GROUPS:
        a, b = grp["a"], grp["b"]
        cs = {a: concept(a), b: concept(b)}
        if any(v is None for v in cs.values()):
            continue
        guide = ConceptSubspace.blend(list(cs.values()))
        for lam in LAMBDAS:
            torch.cuda.empty_cache()
            texts, probe = sur.generate_with_topk_guide(
                prompts, guide=guide, k=K, steps=STEPS, fluency_weight=lam
            )
            harness.evaluate(
                prompts,
                texts,
                [a, b],
                config={
                    "mode": "blend",
                    "model": MODEL_KEY,
                    "arm": arm,
                    "lam": lam,
                    "a": a,
                    "b": b,
                    "pair_rho": grp.get("pair_rho"),
                    "rank": RANK,
                    "r_lemma": R_LEMMA,
                    "k": K,
                    "steps": STEPS,
                },
                probe=probe,
                baseline_texts=baseline_texts,
                concepts=cs,
                angles=False,
            )
    print(f"{arm}: done")

## A. Singles — pooled vs hierarchical (per-concept)

In [ ]:
records = [json.loads(x) for x in LOG_PATH.read_text().strip().split("\n")]

srows = []
for r in records:
    if r["config"]["mode"] != "single":
        continue
    name = r["config"]["concept"]
    srows.append(
        {
            "arm": r["config"]["arm"],
            "lam": r["config"]["lam"],
            "concept": name,
            "delta": r["expression"][name]["delta"],
            "success": r["success"][name],
            "ppl_ratio": r["ppl_ratio"],
        }
    )
sdf = pd.DataFrame(srows)

singles = (
    sdf.groupby(["arm", "lam"])
    .agg(
        delta=("delta", "mean"),
        positive=("delta", lambda s: (s > 0).mean()),
        success=("success", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n=("delta", "size"),
    )
    .reset_index()
)
singles.to_csv(f"resources/57_subspace_ml_singles_summary{SUFFIX}.csv", index=False)
print(singles.round(4).to_string(index=False))

# Per-concept construction comparison at λ=0 (the base steering).
p0 = sdf[sdf.lam == 0.0]
paired = p0.pivot_table(index="concept", columns="arm", values="delta").reset_index()
if {"pooled", "hierarchical"}.issubset(paired.columns):
    paired["hier - pooled"] = paired["hierarchical"] - paired["pooled"]
paired

## B. Blend pairs — multi-concept composition (joint metric, λ dial)

Mirrors the FRH multi-concept steering summary: **both-improved** rate (all members Δ>0) and **joint Δ** (`min` over members), grouped by construction × λ.

In [ ]:
brows = []
for r in records:
    if r["config"]["mode"] != "blend":
        continue
    cfg = r["config"]
    members = [cfg["a"], cfg["b"]]
    deltas = [r["expression"][m]["delta"] for m in members]
    brows.append(
        {
            "arm": cfg["arm"],
            "lam": cfg["lam"],
            "a": cfg["a"],
            "b": cfg["b"],
            "joint_delta": min(deltas),
            "both_improved": all(d > 0 for d in deltas),
            "ppl_ratio": r["ppl_ratio"],
        }
    )
bdf = pd.DataFrame(brows)

blend = (
    bdf.groupby(["arm", "lam"])
    .agg(
        joint_delta=("joint_delta", "mean"),
        both_improved=("both_improved", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n=("joint_delta", "size"),
    )
    .reset_index()
)
blend.to_csv(f"resources/57_subspace_ml_blend_summary{SUFFIX}.csv", index=False)
print(blend.round(4).to_string(index=False))

In [ ]:
lam_colors = ["tab:gray", "tab:blue"]


def grouped_bars(summary, metrics, title, path):
    """Grouped bars over ARMS (x) x LAMBDAS (color) for each metric."""
    x = range(len(ARMS))
    width = 0.38
    fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 4))
    for ax, (col, label) in zip(axes, metrics):
        for j, lam in enumerate(LAMBDAS):
            vals = [
                summary.loc[(summary.arm == a) & (summary.lam == lam), col]
                for a in ARMS
            ]
            vals = [v.iloc[0] if len(v) else float("nan") for v in vals]
            ax.bar(
                [i + (j - 0.5) * width for i in x],
                vals,
                width,
                label=f"λ={lam}",
                color=lam_colors[j],
            )
        ax.set_xticks(list(x))
        ax.set_xticklabels(ARMS, rotation=15)
        ax.set_ylabel(label)
        ax.axhline(0, color="gray", lw=0.6)
        ax.legend()
    axes[-1].axhline(2.5, color="red", ls=":", lw=0.8)  # fluency guardrail
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()


grouped_bars(
    singles,
    [
        ("delta", "mean expression Δ"),
        ("positive", "positive-Δ rate"),
        ("success", "presence-success rate"),
        ("ppl_ratio", "perplexity ratio"),
    ],
    "A. Singles — pooled vs hierarchical under the λ dial",
    f"resources/57_subspace_ml_singles{SUFFIX}.png",
)

grouped_bars(
    blend,
    [
        ("both_improved", "both-improved rate"),
        ("joint_delta", "joint Δ (min over pair)"),
        ("ppl_ratio", "perplexity ratio"),
    ],
    "B. Blend pairs — composition under the λ dial",
    f"resources/57_subspace_ml_blend{SUFFIX}.png",
)

## Notes

- Both constructions share rank `r` and scoring (`‖Bᵀh‖`); any gap is purely the
  concept build (token-vote vs lemma-vote).
- **Singles** use the plain single-guide loop; **blend** merges a pair into one
  subspace (`ConceptSubspace.blend`) — Family-1 composition — so multi-concept
  steering reuses the same loop. `λ` (`fluency_weight`) trades steering strength
  against fluency, matching the FRH multi-concept experiments (NB 40/43).
- Single-vector (SM) scoring only; beam, vote/schedule aggregators, and the MSM
  subspace query are frame-pipeline / follow-up features.
- `SMOKE = True` writes `_smoke` outputs; flip to `False` for the full pool +
  all 10 pairs.